In [ ]:
import requests
import json
import base64

import os
import time

import json
import re

EXPECTED_KEYS = {
    "serial_number",
    "name",
    "father_name",
    "house_number",
    "age",
    "gender",
    "id_number",
    "under_adjudication",
    "file_identifier",
    "file_name"
}

openrouter_api_key='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'
namecard_info_extractor_prompt='''
Extract the required fields from the image.

Fields:
- serial_number
- name
- father_name
- house_number
- age
- gender
- id_number
- under_adjudication

If watermark "Under Adjudication" is visible (even faint):
    under_adjudication = true
Else:
    under_adjudication = false


Return the extracted values.
No markdown.
No explanation.
'''


In [ ]:

def normalize_key(key):
    # Remove surrounding quotes and spaces
    key = key.strip().strip('"').strip("'")
    return key

def normalize_value(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, int):
        return value

    if not isinstance(value, str):
        return value

    value = value.strip()

    # Remove trailing comma
    if value.endswith(","):
        value = value[:-1]

    # Remove surrounding quotes
    value = value.strip().strip('"').strip("'")

    # Convert boolean strings
    if value.lower() == "true":
        return True
    if value.lower() == "false":
        return False

    # Convert integers
    if value.isdigit():
        return int(value)

    return value

def is_valid_json(text):
    try:
        json.loads(text)
        return True
    except:
        return False
    
def clean_answer_list(answer_list):
    cleaned = []
    skipped = []

    for item in answer_list:
        new_item = {}

        for key, value in item.items():

            # 🔥 Case 1: JSON split across key + value
            if key.strip().startswith('{"'):
                reconstructed = key + ":" + str(value)

                if is_valid_json(reconstructed):
                    parsed = json.loads(reconstructed)
                    new_item.update(parsed)
                else:
                    skipped.append(item)
                    continue

            else:
                new_item[key.strip().strip('"')] = value

        # Only keep if it has required fields
        if "serial_number" in new_item and "name" in new_item:
            cleaned.append(new_item)
        else:
            skipped.append(item)

    print(f"Skipped {len(skipped)} malformed entries")

    return cleaned

def parse_key_value_text(text):
    result = {}

    lines = text.strip().split("\n")

    for line in lines:
        if ":" in line:
            key, value = line.split(":", 1)
            key = key.strip()
            value = value.strip()

            # Convert booleans
            if value.lower() == "true":
                value = True
            elif value.lower() == "false":
                value = False

            # Convert integers
            elif value.isdigit():
                value = int(value)

            result[key] = value

    return result

def namecard_info_extract(openrouter_api_key,namecard_info_extractor_prompt,data_url):
    response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {openrouter_api_key}",
        "Content-Type": "application/json",    
    },
    data=json.dumps({
        "model": "google/gemini-2.5-flash",
        "messages": [
        {
            "role": "user",
            "content": [
            {
                "type": "text",
                "text": namecard_info_extractor_prompt
            },
            {
                "type": "image_url",
                "image_url": {
                "url": data_url
                }
            }
            
            ]
        }
        ]
    })
    )    
    content = response.json()["choices"][0]["message"]["content"]
    parsed_dict = parse_key_value_text(content)
    return parsed_dict
    

In [ ]:
file_identifier='2026-EROLLGEN-S25-161-SIR-FinalRoll-Revision1-ENG-136-WI'
base_folder=f'/Users/ashhadulislam/projects/social/SIR/namecard_sets/{file_identifier}'


answer_list=[]
counter=0
for f in os.listdir(base_folder):
    #print(f)
    image_file_location=f'{base_folder}/{f}'

    
    with open(image_file_location, "rb") as img_file:
        base64_image = base64.b64encode(img_file.read()).decode("utf-8")

    data_url = f"data:image/png;base64,{base64_image}"
    answer=namecard_info_extract(openrouter_api_key,namecard_info_extractor_prompt,data_url)
    answer['file_identifier']=file_identifier
    answer['file_name']=f
    answer_list.append(answer)
    counter+=1

    if counter%50==0:
        print(counter,'/',len(os.listdir(base_folder)),time.time())



import pandas as pd
cleaned_answer_list = clean_answer_list(answer_list)

df=pd.DataFrame(cleaned_answer_list)
df=df.sort_values(by=['file_name'])
save_folder='/Users/ashhadulislam/projects/social/SIR/output_sheets'
df.to_csv(f'{save_folder}/{file_identifier}.csv',index=False)
    